<a href="https://colab.research.google.com/github/pasupunoornishita/Realistic-Human-Face-Generation-Using-Gan-s/blob/main/face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision flask-ngrok flask pillow matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
from flask import Flask, request, send_file
from flask_ngrok import run_with_ngrok
import os

In [ ]:
import zipfile
with zipfile.ZipFile("final_celeba.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [ ]:
# Adjust this path to your Colab dataset
dataset_path = "final_celeba/face"

# Create 'all_faces' subfolder if not exists
all_face_path = os.path.join(dataset_path, "all_face")
os.makedirs(all_face_path, exist_ok=True)

# Move all .jpg images into 'all_faces'
for f in os.listdir(dataset_path):
    src = os.path.join(dataset_path, f)
    dst = os.path.join(all_face_path, f)
    if os.path.isfile(src) and f.lower().endswith(".jpg"):
        if not os.path.exists(dst):
            os.rename(src, dst)

# Transform: resize + normalize to [-1,1]
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# Load dataset
dataset = datasets.ImageFolder(dataset_path, transform=transform)

# Split into train/test (80/20)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✅ Dataset ready. Train: {len(train_dataset)}, Test: {len(test_dataset)}")

✅ Dataset ready. Train: 12216, Test: 3054


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0),  # 1x1 -> 4x4
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 4x4 -> 8x8
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 8x8 -> 16x16
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 16x16 -> 32x32
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),     # 32x32 -> 64x64
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),    # 64x64 -> 32x32
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),  # 32x32 -> 16x16
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1), # 16x16 -> 8x8
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1), # 8x8 -> 4x4
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 0),   # 4x4 -> 1x1
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).view(x.size(0))

In [ ]:
import torch
import torch.nn as nn

# ====== GLOBAL CONFIG ======
latent_dim = 100        # MUST exist before Generator()
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
G = Generator(latent_dim).to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
lr = 0.0002

optimizer_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5,0.999))
optimizer_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5,0.999))

print(f"✅ GAN Initialized. Device: {device}")

✅ GAN Initialized. Device: cuda


In [ ]:
num_epochs = 50

for epoch in range(num_epochs):
    for i, (imgs, _) in enumerate(train_loader):
        real_imgs = imgs.to(device)
        batch_size = real_imgs.size(0)

        # Real & fake labels
        real = torch.ones(batch_size, device=device)
        fake = torch.zeros(batch_size, device=device)

        # -------------------------
        # Train Generator
        # -------------------------
        optimizer_G.zero_grad()
        z = torch.randn(batch_size, latent_dim, 1, 1, device=device)
        gen_imgs = G(z)
        g_loss = criterion(D(gen_imgs), real)
        g_loss.backward()
        optimizer_G.step()

        # -------------------------
        # Train Discriminator
        # -------------------------
        optimizer_D.zero_grad()
        real_loss = criterion(D(real_imgs), real)
        fake_loss = criterion(D(gen_imgs.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

    print(f"Epoch [{epoch+1}/{num_epochs}]  D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

Epoch [1/50]  D_loss: 0.3417  G_loss: 3.2030
Epoch [2/50]  D_loss: 0.1575  G_loss: 2.0438
Epoch [3/50]  D_loss: 0.1203  G_loss: 3.3279
Epoch [4/50]  D_loss: 0.1475  G_loss: 1.8472
Epoch [5/50]  D_loss: 0.2094  G_loss: 3.5970
Epoch [6/50]  D_loss: 0.1418  G_loss: 4.0947
Epoch [7/50]  D_loss: 0.0550  G_loss: 4.6123
Epoch [8/50]  D_loss: 0.0781  G_loss: 3.0768
Epoch [9/50]  D_loss: 0.3519  G_loss: 3.4350
Epoch [10/50]  D_loss: 0.0826  G_loss: 5.1428
Epoch [11/50]  D_loss: 0.1311  G_loss: 2.5707
Epoch [12/50]  D_loss: 0.1732  G_loss: 4.8428
Epoch [13/50]  D_loss: 0.0956  G_loss: 2.6004
Epoch [14/50]  D_loss: 0.2748  G_loss: 2.6725
Epoch [15/50]  D_loss: 0.2530  G_loss: 1.4428
Epoch [16/50]  D_loss: 0.0907  G_loss: 3.0274
Epoch [17/50]  D_loss: 0.1963  G_loss: 3.5659
Epoch [18/50]  D_loss: 0.0336  G_loss: 3.5560
Epoch [19/50]  D_loss: 0.0806  G_loss: 2.5062
Epoch [20/50]  D_loss: 0.0846  G_loss: 3.0049
Epoch [21/50]  D_loss: 0.3088  G_loss: 5.6397
Epoch [22/50]  D_loss: 0.1858  G_loss: 3.43

In [ ]:
torch.save(G.state_dict(), "generator.pth")

In [ ]:
import gradio as gr
import torch
from torchvision import transforms

In [ ]:
# Step 3: Function to modify latent vector based on attributes
def apply_attributes(z, age, smile, hair, gender):
    # age affects channel 0
    z[:,0,:,:] += (age - 50) / 50 * 0.2

    # smile affects channel 1
    z[:,1,:,:] += 0.1 if smile=="yes" else -0.1

    # hair affects channel 2
    hair_map = {"black":0, "brown":0.1, "blonde":0.2, "other":-0.1}
    z[:,2,:,:] += hair_map.get(hair,0)

    # gender affects channel 3
    z[:,3,:,:] += 0.1 if gender=="female" else -0.1

    return z

# Step 4: Function to generate face
def generate_face(age, smile, hair, gender):
    z = torch.randn(1, latent_dim, 1, 1, device=device)
    z = apply_attributes(z, age, smile, hair, gender)
    img_tensor = G(z)[0]
    img_tensor = (img_tensor + 1)/2  # [-1,1] -> [0,1]
    img = transforms.ToPILImage()(img_tensor.cpu())

    img = img.resize((64, 64))
    return img

# Step 5: Gradio Interface
iface = gr.Interface(
    fn=generate_face,
    inputs=[
        gr.Slider(0, 100, value=25, label="Age"),
        gr.Dropdown(["yes","no"], value="yes", label="Smile"),
        gr.Dropdown(["black","brown","blonde","other"], value="black", label="Hair"),
        gr.Dropdown(["male","female"], value="male", label="Gender")  # <-- Added Gender
    ],
    outputs="image",
    title="GAN Face Generator",
    description="Generate a face based on Age, Smile, Hair, and Gender"
)

# Step 6: Launch Gradio
iface.launch()

NameError: name 'gr' is not defined

In [ ]:
!pip install torch torchvision gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9

In [ ]:
import torch
import torchvision.utils as vutils
from PIL import Image
import gradio as gr